# 10_業務_月營收趨勢

折線圖與月增率

任務：依月份排序，確認沒有缺月或重複月份。；畫折線圖，標示月份、金額單位與標題。；算 2～6 月月增率，第一個月保留空白。

規則：月增率＝（本月－上月）÷ 上月；上月若為 0，應標示無法計算；先排序再比較。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('10_業務_月營收趨勢.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '10_業務_月營收趨勢'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
import matplotlib.pyplot as plt
result = sorted(rows, key=lambda r: r['月份'])
assert [r['月份'] for r in result] == [f'2025-{m:02d}' for m in range(1,7)]
amounts = [r['營收_元'] for r in result]
growth = [None] + [(b-a)/a if a != 0 else None for a,b in zip(amounts,amounts[1:])]
print('月增率：', growth)
fig, ax = plt.subplots()
ax.plot([r['月份'] for r in result], amounts, marker='o')
ax.set(title='Monthly revenue', xlabel='Month', ylabel='Revenue (TWD)')
fig.tight_layout()
fig.savefig(OUT / '月營收.png', dpi=150)
plt.close(fig)
assert sum(amounts) == 800000 and growth[2] == -0.25
wb.close()

## 驗收答案

半年營收 800,000 元；3 月月增率 −25%；6 月較 5 月增加 12.5%。